# Pilot 5 · Readable traces of rank-1 SFT and GRPO

Bootstrap notebook · implementation in progress. Reuses saved adapters; no training runs here.

Run **1–2** after every runtime reset. Run **3** to audit selected sources. For the minimal Colab smoke, use **4–5**, restart the runtime, then **1–2 and 6**. Do not use Run all.


## 1. Mount storage and load the pinned harness

CPU or GPU. The harness has a separate checkout; your saved experiment artifacts stay in Drive.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts')
EXPERIMENT_NAME = 'pilot5-readable-traces-v1'
HARNESS_COMMIT = 'dc8255f'
REPO_DIR = Path('/content/pilot5-harness') / HARNESS_COMMIT

if not (REPO_DIR / '.git').is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/mahadikprasad15/spar-pilot.git',
                    str(REPO_DIR)], check=True)
    subprocess.run(['git', 'checkout', '--detach', HARNESS_COMMIT], cwd=REPO_DIR, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', '--short=7', 'HEAD'],
                                        cwd=REPO_DIR, text=True).strip()
assert actual_commit == HARNESS_COMMIT, 'Harness checkout differs from its pin.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


## 2. Restore saved state

Read-only. No model imports or earlier experiment stages run. “Audited config” records a previous source audit; capture will revalidate the source evidence.


In [ ]:
from pilot_eval.trace_session import TraceSession

session = TraceSession(ARTIFACT_ROOT, REPO_DIR, EXPERIMENT_NAME)
state = session.restore()
print(json.dumps(state, indent=2))


## 3. Select and audit existing sources (CPU)

These are the existing prepared source configurations. Change a path only when selecting another reviewed source. Auditing checks their existing seals and source evidence; it does not load weights or generate answers.


In [ ]:
SOURCE_CONFIGS = {
    'sft': ARTIFACT_ROOT / 'plans/pilot3-inputs-v1/activation.prepared.json',
    'grpo': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
    'random': ARTIFACT_ROOT / 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
}
session.select_sources(SOURCE_CONFIGS)
source_audit = session.audit_sources()
print(json.dumps(source_audit, indent=2))


## 4. Install and check pinned GPU dependencies

GPU smoke only. Removes conflicting optional vision/audio packages and imports the actual text-model classes in a fresh process. No model weights are downloaded. If Colab requests a restart, run 1–2 again before continuing.


In [ ]:
from pilot_eval.notebook_session import NotebookSession

environment_setup = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=EXPERIMENT_NAME)
environment_setup.install_environment()
runtime = session.readiness()
print(json.dumps(runtime, indent=2))


## 5. Write the minimal Colab smoke artifact

Records actual import-check duration and GPU allocation, writes a small sealed artifact to Drive, then reads it back. After this cell finishes, restart the runtime. The GPU is not needed for cell 6.


In [ ]:
smoke = session.write_smoke()
print(json.dumps(smoke, indent=2))
print('Saved smoke:', session.directory)
print('Next: restart runtime, run 1–2, then 6.')


## 6. Verify storage after reconnect (CPU)

Run only in the fresh kernel after 1–2. Verifies the saved smoke seal and records a reconnect receipt. This qualifies bootstrap imports/storage only; capture and lens qualification are later tickets.


In [ ]:
restored = session.verify_reconnect()
print(json.dumps(restored['smoke'], indent=2))
print('Real-Colab bootstrap validation:', restored['target_validation'])


## Next stages

Corpus preparation, capture qualification and community-lens qualification are not implemented in this notebook yet. Scientific settings remain subject to their review gates. Save the smoke results before the next implementation slice.
